# Football Match Outcome Prediction — revised evaluation

Three-class Premier League prediction (home win, draw, away win). This revision retains the original feature engineering but separates chronological training, validation and final testing, and fits missing-value preprocessing within the training/CV pipeline. Historical test results from the original notebook have already been inspected, so a genuinely unseen later season is necessary for a new independent benchmark. Lineup-based experiments assume the starting lineup is available at prediction time.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from pathlib import Path
from sklearn.base import clone
from sklearn.impute import SimpleImputer

In [ ]:
db_path = Path("database.sqlite").resolve()
if not db_path.is_file():
    raise FileNotFoundError("Place the original football database.sqlite next to this notebook.")
conn = sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True)

## Data extraction and chronological history

In [ ]:
query = """
SELECT
    m.*,
    ht.team_long_name AS home_team,
    at.team_long_name AS away_team
FROM Match AS m
JOIN League AS l
    ON m.league_id = l.id
JOIN Team AS ht
    ON m.home_team_api_id = ht.team_api_id
JOIN Team AS at
    ON m.away_team_api_id = at.team_api_id
WHERE l.name = 'England Premier League'
ORDER BY m.date;
"""

matches = pd.read_sql(query, conn)

In [ ]:
matches["date"] = pd.to_datetime(matches["date"])
matches = matches.sort_values("date").reset_index(drop=True)
matches = matches.dropna(
    subset=["home_team_goal", "away_team_goal"]
)
matches = matches.drop_duplicates(subset=["match_api_id"]).reset_index(drop=True)

In [ ]:
def create_result(row):
    if row["home_team_goal"] > row["away_team_goal"]:
        return "H"
    elif row["home_team_goal"] < row["away_team_goal"]:
        return "A"
    return "D"

matches["result"] = matches.apply(create_result, axis=1)


## Experiment 1 — recent team form
Rolling statistics exclude the current match via shift(1).

In [ ]:
# Build team-level match history from the perspective of each team

home_history = matches[[
    "match_api_id", "date", "season",
    "home_team_api_id", "away_team_api_id",
    "home_team", "away_team",
    "home_team_goal", "away_team_goal"
]].copy()

home_history = home_history.rename(columns={
    "home_team_api_id": "team_api_id",
    "away_team_api_id": "opponent_api_id",
    "home_team": "team_name",
    "away_team": "opponent_name",
    "home_team_goal": "goals_for",
    "away_team_goal": "goals_against"
})

home_history["is_home"] = 1


away_history = matches[[
    "match_api_id", "date", "season",
    "away_team_api_id", "home_team_api_id",
    "away_team", "home_team",
    "away_team_goal", "home_team_goal"
]].copy()

away_history = away_history.rename(columns={
    "away_team_api_id": "team_api_id",
    "home_team_api_id": "opponent_api_id",
    "away_team": "team_name",
    "home_team": "opponent_name",
    "away_team_goal": "goals_for",
    "home_team_goal": "goals_against"
})

away_history["is_home"] = 0


team_history = pd.concat([home_history, away_history], ignore_index=True)
team_history = team_history.sort_values(["team_api_id", "date"]).reset_index(drop=True)

team_history.head()

In [ ]:
def calculate_points(row):
    if row["goals_for"] > row["goals_against"]:
        return 3
    elif row["goals_for"] == row["goals_against"]:
        return 1
    else:
        return 0

team_history["points"] = team_history.apply(calculate_points, axis=1)
team_history["win"] = (team_history["points"] == 3).astype(int)
team_history["draw"] = (team_history["points"] == 1).astype(int)
team_history["loss"] = (team_history["points"] == 0).astype(int)

team_history[[
    "date", "team_name", "opponent_name",
    "is_home", "goals_for", "goals_against",
    "points", "win", "draw", "loss"
]].head()

In [ ]:
team_history = team_history.sort_values(["team_api_id", "date"]).reset_index(drop=True)

team_history["points_last5"] = (
    team_history
    .groupby("team_api_id")["points"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)

team_history["goals_for_last5"] = (
    team_history
    .groupby("team_api_id")["goals_for"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
# یعنی بازی فعلی را وارد محاسبه نکن
team_history["goals_against_last5"] = (
    team_history
    .groupby("team_api_id")["goals_against"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

team_history["wins_last5"] = (
    team_history
    .groupby("team_api_id")["win"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)

team_history["draws_last5"] = (
    team_history
    .groupby("team_api_id")["draw"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)

team_history["losses_last5"] = (
    team_history
    .groupby("team_api_id")["loss"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)

team_history["previous_match_date"] = (
    team_history
    .groupby("team_api_id")["date"]
    .shift(1)
)

team_history["rest_days"] = (
    team_history["date"] - team_history["previous_match_date"]
).dt.days

team_history.head()

In [ ]:
form_columns = [
    "match_api_id",
    "points_last5",
    "goals_for_last5",
    "goals_against_last5",
    "wins_last5",
    "draws_last5",
    "losses_last5",
    "rest_days"
]

home_form = team_history[team_history["is_home"] == 1][form_columns].copy()
away_form = team_history[team_history["is_home"] == 0][form_columns].copy()

home_form = home_form.rename(columns={
    "points_last5": "home_points_last5",
    "goals_for_last5": "home_goals_for_last5",
    "goals_against_last5": "home_goals_against_last5",
    "wins_last5": "home_wins_last5",
    "draws_last5": "home_draws_last5",
    "losses_last5": "home_losses_last5",
    "rest_days": "home_rest_days"
})

away_form = away_form.rename(columns={
    "points_last5": "away_points_last5",
    "goals_for_last5": "away_goals_for_last5",
    "goals_against_last5": "away_goals_against_last5",
    "wins_last5": "away_wins_last5",
    "draws_last5": "away_draws_last5",
    "losses_last5": "away_losses_last5",
    "rest_days": "away_rest_days"
})

df = matches.merge(home_form, on="match_api_id", how="left")
df = df.merge(away_form, on="match_api_id", how="left")

df.head()

In [ ]:
df["form_difference"] = df["home_points_last5"] - df["away_points_last5"]

df["attack_difference"] = (
    df["home_goals_for_last5"] -
    df["away_goals_for_last5"]
)
# اینجا برعکس کم کردن را طوری نوشتیم که اگر عدد مثبت شود، یعنی دفاع میزبان بهتر بوده
df["defense_difference"] = (
    df["away_goals_against_last5"] -
    df["home_goals_against_last5"]
)

df["win_difference"] = df["home_wins_last5"] - df["away_wins_last5"]

df["rest_difference"] = df["home_rest_days"] - df["away_rest_days"]

df[[
    "home_team", "away_team", "result",
    "home_points_last5", "away_points_last5",
    "form_difference",
    "attack_difference",
    "defense_difference"
]].head()

In [ ]:
form_feature_columns = [
    "home_points_last5",
    "away_points_last5",
    "home_goals_for_last5",
    "away_goals_for_last5",
    "home_goals_against_last5",
    "away_goals_against_last5",
    "home_wins_last5",
    "away_wins_last5",
    "home_draws_last5",
    "away_draws_last5",
    "home_losses_last5",
    "away_losses_last5",
    "home_rest_days",
    "away_rest_days",
    "form_difference",
    "attack_difference",
    "defense_difference",
    "win_difference",
    "rest_difference"
]

df[form_feature_columns] = df[form_feature_columns].fillna(0)

df[form_feature_columns].isnull().sum()

In [ ]:
feature_columns_base = [
    "home_points_last5",
    "away_points_last5",
    "home_goals_for_last5",
    "away_goals_for_last5",
    "home_goals_against_last5",
    "away_goals_against_last5",
    "home_wins_last5",
    "away_wins_last5",
    "home_rest_days",
    "away_rest_days",
    "form_difference",
    "attack_difference",
    "defense_difference",
    "win_difference",
    "rest_difference"
]

model_df_base = df[[
    "date", "season", "home_team", "away_team", "result"
] + feature_columns_base].copy()

model_df_base = model_df_base.sort_values("date").reset_index(drop=True)

model_df_base.head()

## Experiment 2 — team form and starting-lineup ratings
Player ratings are joined from observations strictly before the match timestamp; missing lineup features are left for fold-local imputation. The original position groups are heuristic.

In [ ]:
player_attributes = pd.read_sql("""
SELECT
    player_api_id,
    date,
    overall_rating
FROM Player_Attributes
WHERE overall_rating IS NOT NULL;
""", conn)

player_attributes["date"] = pd.to_datetime(player_attributes["date"])
player_attributes = player_attributes.sort_values(["player_api_id", "date"])

player_attributes.head()

In [ ]:
home_player_cols = [f"home_player_{i}" for i in range(1, 12)]
away_player_cols = [f"away_player_{i}" for i in range(1, 12)]

home_lineup = matches[["match_api_id", "date"] + home_player_cols].copy()
home_lineup = home_lineup.melt(
    id_vars=["match_api_id", "date"],
    value_vars=home_player_cols,
    var_name="player_position",
    value_name="player_api_id"
)
home_lineup["side"] = "home"
home_lineup["position_number"] = home_lineup["player_position"].str.extract(r"(\d+)").astype(int)

away_lineup = matches[["match_api_id", "date"] + away_player_cols].copy()
away_lineup = away_lineup.melt(
    id_vars=["match_api_id", "date"],
    value_vars=away_player_cols,
    var_name="player_position",
    value_name="player_api_id"
)
away_lineup["side"] = "away"
away_lineup["position_number"] = away_lineup["player_position"].str.extract(r"(\d+)").astype(int)

lineup_long = pd.concat([home_lineup, away_lineup], ignore_index=True)
lineup_long["date"] = pd.to_datetime(lineup_long["date"])

lineup_long = lineup_long.dropna(subset=["player_api_id"])
lineup_long["player_api_id"] = lineup_long["player_api_id"].astype(int)

lineup_long.head()

In [ ]:
def add_latest_player_rating(lineup_data, attributes_data):
    result_parts = []

    for player_id, player_matches in lineup_data.groupby("player_api_id"):
        player_matches = player_matches.sort_values("date")

        player_attr = attributes_data[
            attributes_data["player_api_id"] == player_id
        ].sort_values("date")

        if player_attr.empty:
            player_matches["overall_rating"] = np.nan
            result_parts.append(player_matches)
            continue

        merged = pd.merge_asof(
            player_matches,
            player_attr[["date", "overall_rating"]],
            on="date",
            direction="backward",
            allow_exact_matches=False
        )

        result_parts.append(merged)

    return pd.concat(result_parts, ignore_index=True)

lineup_with_rating = add_latest_player_rating(lineup_long, player_attributes)

lineup_with_rating.head()

In [ ]:
def position_group(position_number):
    if position_number == 1:
        return "goalkeeper"
    elif position_number <= 5:
        return "defense"
    elif position_number <= 8:
        return "midfield"
    else:
        return "attack"

lineup_with_rating["position_group"] = lineup_with_rating["position_number"].apply(position_group)

lineup_with_rating[[
    "match_api_id", "side", "position_number",
    "position_group", "player_api_id", "overall_rating"
]].head()

In [ ]:
lineup_summary = (
    lineup_with_rating
    .groupby(["match_api_id", "side"])
    .agg(
        lineup_avg_rating=("overall_rating", "mean"),
        lineup_max_rating=("overall_rating", "max"),
        lineup_min_rating=("overall_rating", "min"),
        known_players=("overall_rating", "count")
    )
    .reset_index()
)

lineup_summary.head()

In [ ]:
line_group_summary = (
    lineup_with_rating
    .groupby(["match_api_id", "side", "position_group"])["overall_rating"]
    .mean()
    .reset_index()
)

line_group_pivot = line_group_summary.pivot_table(
    index=["match_api_id", "side"],
    columns="position_group",
    values="overall_rating"
).reset_index()

lineup_features = lineup_summary.merge(
    line_group_pivot,
    on=["match_api_id", "side"],
    how="left"
)

lineup_features.head()

In [ ]:
home_lineup_features = lineup_features[lineup_features["side"] == "home"].copy()
away_lineup_features = lineup_features[lineup_features["side"] == "away"].copy()

home_lineup_features = home_lineup_features.drop(columns=["side"])
away_lineup_features = away_lineup_features.drop(columns=["side"])

home_lineup_features = home_lineup_features.add_prefix("home_")
away_lineup_features = away_lineup_features.add_prefix("away_")

home_lineup_features = home_lineup_features.rename(columns={
    "home_match_api_id": "match_api_id"
})

away_lineup_features = away_lineup_features.rename(columns={
    "away_match_api_id": "match_api_id"
})

df2 = df.merge(home_lineup_features, on="match_api_id", how="left")
df2 = df2.merge(away_lineup_features, on="match_api_id", how="left")

df2.head()

In [ ]:
df2["lineup_rating_difference"] = (
    df2["home_lineup_avg_rating"] -
    df2["away_lineup_avg_rating"]
)

df2["attack_rating_difference"] = (
    df2["home_attack"] -
    df2["away_attack"]
)

df2["midfield_rating_difference"] = (
    df2["home_midfield"] -
    df2["away_midfield"]
)

df2["defense_rating_difference"] = (
    df2["home_defense"] -
    df2["away_defense"]
)

df2["goalkeeper_rating_difference"] = (
    df2["home_goalkeeper"] -
    df2["away_goalkeeper"]
)

lineup_feature_columns = [
    "home_lineup_avg_rating",
    "away_lineup_avg_rating",
    "home_lineup_max_rating",
    "away_lineup_max_rating",
    "home_known_players",
    "away_known_players",
    "home_goalkeeper",
    "away_goalkeeper",
    "home_defense",
    "away_defense",
    "home_midfield",
    "away_midfield",
    "home_attack",
    "away_attack",
    "lineup_rating_difference",
    "attack_rating_difference",
    "midfield_rating_difference",
    "defense_rating_difference",
    "goalkeeper_rating_difference"
]

df2[lineup_feature_columns].head()

In [ ]:
feature_columns_full = feature_columns_base + lineup_feature_columns

model_df_full = df2[[
    "date", "season", "home_team", "away_team", "result"
] + feature_columns_full].copy()

model_df_full = model_df_full.sort_values("date").reset_index(drop=True)

model_df_full.head()

## Experiment 3 — add head-to-head history
Only earlier matches between the same teams are used.

In [ ]:
# ============================================================
# Experiment 3: Add Head-to-Head Features
# ============================================================

matches_h2h = matches[[
    "match_api_id",
    "date",
    "home_team_api_id",
    "away_team_api_id",
    "home_team_goal",
    "away_team_goal"
]].copy()

matches_h2h["date"] = pd.to_datetime(matches_h2h["date"])
matches_h2h = matches_h2h.sort_values("date").reset_index(drop=True)


def calculate_h2h_features(matches_data, last_n=5):
    """
    Create Head-to-Head features for each match.
    For each match, only previous matches between the same two teams are used.
    This avoids data leakage.
    """

    h2h_rows = []

    matches_data = matches_data.sort_values("date").reset_index(drop=True)

    for idx, row in matches_data.iterrows():
        current_match_id = row["match_api_id"]
        current_date = row["date"]
        current_home = row["home_team_api_id"]
        current_away = row["away_team_api_id"]

        previous_matches = matches_data[
            (
                (
                    (matches_data["home_team_api_id"] == current_home) &
                    (matches_data["away_team_api_id"] == current_away)
                )
                |
                (
                    (matches_data["home_team_api_id"] == current_away) &
                    (matches_data["away_team_api_id"] == current_home)
                )
            )
            &
            (matches_data["date"] < current_date)
        ].sort_values("date")

        previous_matches = previous_matches.tail(last_n)

        home_points = []
        away_points = []
        goal_diffs = []

        home_wins = 0
        draws = 0
        away_wins = 0

        for _, prev in previous_matches.iterrows():

            # Convert previous match result to the perspective of the CURRENT home team
            if prev["home_team_api_id"] == current_home:
                current_home_goals = prev["home_team_goal"]
                current_away_goals = prev["away_team_goal"]
            else:
                current_home_goals = prev["away_team_goal"]
                current_away_goals = prev["home_team_goal"]

            if current_home_goals > current_away_goals:
                home_points.append(3)
                away_points.append(0)
                home_wins += 1
            elif current_home_goals < current_away_goals:
                home_points.append(0)
                away_points.append(3)
                away_wins += 1
            else:
                home_points.append(1)
                away_points.append(1)
                draws += 1

            goal_diffs.append(current_home_goals - current_away_goals)

        h2h_rows.append({
            "match_api_id": current_match_id,
            "h2h_matches_before": len(previous_matches),
            "h2h_home_points_last5": np.sum(home_points) if len(home_points) > 0 else 0,
            "h2h_away_points_last5": np.sum(away_points) if len(away_points) > 0 else 0,
            "h2h_goal_diff_last5": np.mean(goal_diffs) if len(goal_diffs) > 0 else 0,
            "h2h_home_wins_last5": home_wins,
            "h2h_draws_last5": draws,
            "h2h_away_wins_last5": away_wins
        })

    return pd.DataFrame(h2h_rows)


h2h_features = calculate_h2h_features(matches_h2h, last_n=5)

h2h_features.head()

In [ ]:
# Merge Head-to-Head features with the dataset that already contains form + lineup features

df3 = df2.merge(h2h_features, on="match_api_id", how="left")

h2h_feature_columns = [
    "h2h_matches_before",
    "h2h_home_points_last5",
    "h2h_away_points_last5",
    "h2h_goal_diff_last5",
    "h2h_home_wins_last5",
    "h2h_draws_last5",
    "h2h_away_wins_last5"
]

df3[h2h_feature_columns] = df3[h2h_feature_columns].fillna(0)

df3[h2h_feature_columns].head()

In [ ]:
# Final feature set for Experiment 3

feature_columns_full_h2h = (
    feature_columns_base +
    lineup_feature_columns +
    h2h_feature_columns
)

model_df_h2h = df3[[
    "date", "season", "home_team", "away_team", "result"
] + feature_columns_full_h2h].copy()

model_df_h2h = model_df_h2h.sort_values("date").reset_index(drop=True)

model_df_h2h.head()

## Model selection on validation season
Train 2008/09–2013/14; validation 2014/15; final historical holdout 2015/16. Macro F1 is the selection criterion. Earlier match outcomes within validation/test seasons may contribute to later matches, consistent with sequential match-by-match prediction, not a season-ahead forecast.

In [ ]:
models = {
    "Dummy Baseline": DummyClassifier(strategy="most_frequent"),
    "KNN": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=5))]),
    "Logistic Regression": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))]),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_split=5, random_state=42, class_weight="balanced"),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

experiments = {
    "Team Form": (model_df_base, feature_columns_base),
    "Team Form + Lineup": (model_df_full, feature_columns_full),
    "Team Form + Lineup + H2H": (model_df_h2h, feature_columns_full_h2h),
}
train_seasons = ["2008/2009", "2009/2010", "2010/2011", "2011/2012", "2012/2013", "2013/2014"]
validation_season, test_season = "2014/2015", "2015/2016"

def partition(frame, columns):
    frame = frame.sort_values("date")
    train = frame[frame["season"].isin(train_seasons)]
    valid = frame[frame["season"] == validation_season]
    test = frame[frame["season"] == test_season]
    assert len(train) and len(valid) and len(test)
    assert train["date"].max() < valid["date"].min() < test["date"].min()
    assert valid["date"].max() < test["date"].min()
    return train, valid, test

def wrapped(estimator):
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("classifier", clone(estimator))])

def scores(y, pred):
    return {"Accuracy": accuracy_score(y, pred), "Macro F1": f1_score(y, pred, average="macro", labels=["H","D","A"], zero_division=0)}

results = []
candidates = {}
for experiment, (frame, cols) in experiments.items():
    train, valid, _ = partition(frame, cols)
    for name, estimator in models.items():
        model = wrapped(estimator)
        model.fit(train[cols], train["result"])
        results.append({"Experiment": experiment, "Model": name, **scores(valid["result"], model.predict(valid[cols]))})
        candidates[(experiment, name)] = clone(model)
pd.DataFrame(results).sort_values("Macro F1", ascending=False)

## Hyperparameter search
TimeSeriesSplit is applied only to training seasons. The imputer is fitted separately in every CV fold. A separate validation season compares tuned and untuned candidates.

In [ ]:
param_grid = {
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__max_depth": [5, 10, 15, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2"],
}
for experiment in ["Team Form + Lineup", "Team Form + Lineup + H2H"]:
    frame, cols = experiments[experiment]
    train, valid, _ = partition(frame, cols)
    # Split on unique match dates so that simultaneous fixtures stay in one fold.
    dates = np.array(sorted(train["date"].unique()))
    folds = []
    for ti, vi in TimeSeriesSplit(n_splits=5).split(dates):
        folds.append((np.flatnonzero(train["date"].isin(dates[ti])), np.flatnonzero(train["date"].isin(dates[vi]))))
    search = RandomizedSearchCV(
        wrapped(RandomForestClassifier(random_state=42, class_weight="balanced")),
        param_distributions=param_grid, n_iter=30, cv=folds,
        scoring="f1_macro", random_state=42, n_jobs=-1,
    )
    search.fit(train[cols], train["result"])
    model = search.best_estimator_
    name = "Tuned Random Forest"
    results.append({"Experiment": experiment, "Model": name, **scores(valid["result"], model.predict(valid[cols]))})
    candidates[(experiment, name)] = clone(model)
    print(experiment, search.best_params_, search.best_score_)

validation_results = pd.DataFrame(results).sort_values(["Macro F1", "Accuracy"], ascending=False)
validation_results

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=validation_results, x="Model", y="Macro F1", hue="Experiment")
plt.xticks(rotation=35, ha="right")
plt.title("Validation Macro F1 by model and feature set")
plt.tight_layout()
plt.show()

## Final historical holdout
Freeze the winning candidate according to validation scores, refit on train + validation, and evaluate the historical test season once. Do not use this result to choose a different model. A new unseen season is needed for independent evaluation because the original experiment already examined this holdout.

In [ ]:
winner = validation_results.iloc[0]
experiment, name = winner["Experiment"], winner["Model"]
frame, cols = experiments[experiment]
train, valid, test = partition(frame, cols)
development = pd.concat([train, valid]).sort_values("date")
final_model = clone(candidates[(experiment, name)])
final_model.fit(development[cols], development["result"])
pred = final_model.predict(test[cols])
print("Selected:", experiment, name)
print(scores(test["result"], pred))
print(classification_report(test["result"], pred, labels=["H","D","A"], zero_division=0))
sns.heatmap(confusion_matrix(test["result"], pred, labels=["H","D","A"]), annot=True, fmt="d", xticklabels=["H","D","A"], yticklabels=["H","D","A"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()
conn.close()